# Hierarchical Architecture — CrewAI

**Pattern:** `Process.hierarchical` — manager LLM orchestrates workers  
**Framework:** CrewAI  

```
┌──────────────────────────────────────────────────┐
│  CREW (Process.hierarchical)                     │
│                                                  │
│           ┌──────────────┐                       │
│           │  manager_llm │  ← auto-created       │
│           └──────┬───────┘    by CrewAI          │
│                  │                               │
│         ┌────────┼────────┐                      │
│         ▼        ▼        ▼                      │
│   Weather     Safety   Report                    │
│   Specialist  Analyst  Writer                    │
└──────────────────────────────────────────────────┘
```

**Key difference from other frameworks:**  
You don't write the manager logic — you set `Process.hierarchical` + `manager_llm`.  
CrewAI injects a manager agent that autonomously plans, delegates, and synthesizes.

In [ ]:
import os
from pydantic import BaseModel, Field
from typing import List
from dotenv import load_dotenv
from crewai import Agent, Task, Crew, Process, LLM
from crewai.tools import tool

load_dotenv()
assert os.getenv("GOOGLE_API_KEY"), "Set GOOGLE_API_KEY in .env"
gemini = LLM(model="gemini/gemini-2.0-flash", temperature=0)
print("✓ Ready")

In [ ]:
@tool("Weather Researcher")
def get_weather(city: str) -> str:
    """Get weather data for a city. Input: city name."""
    data = {"tokyo": "Clear 18°C (9/10)", "paris": "Partly Cloudy 16°C (7/10)", "bangalore": "Rainy 25°C (6/10)"}
    return data.get(city.lower(), f"No data for '{city}'.")

@tool("Safety Researcher")
def get_safety(city: str) -> str:
    """Get safety advisory for a city. Input: city name."""
    data = {"tokyo": "Low (10/10). Very safe.", "paris": "Low (8/10). Alert in crowded spots.", "bangalore": "Medium (6/10). Monsoon affects transport."}
    return data.get(city.lower(), f"No data for '{city}'.")

class CitySection(BaseModel):
    city: str; weather_summary: str; safety_summary: str; recommendation: str

class HierarchicalReport(BaseModel):
    city_sections: List[CitySection]
    executive_summary: str
    top_city: str

print("Tools + schema ready")

In [ ]:
weather_researcher = Agent(
    role="Weather Intelligence Specialist", goal="Collect weather data for cities.",
    backstory="Expert meteorologist who translates weather into traveler reports.",
    tools=[get_weather], llm=gemini, verbose=False,
)
safety_researcher = Agent(
    role="Travel Safety Analyst", goal="Collect safety advisories for cities.",
    backstory="Security analyst specializing in travel safety assessment.",
    tools=[get_safety], llm=gemini, verbose=False,
)
report_writer = Agent(
    role="Travel Report Editor", goal="Synthesize research into polished reports.",
    backstory="Senior travel editor who turns raw research into actionable reports.",
    tools=[], llm=gemini, verbose=False,
)
print("3 worker agents defined")

In [ ]:
cities = ["Tokyo", "Paris"]
cities_str = ", ".join(cities)

weather_task = Task(
    description=f"Get weather for: {cities_str}.",
    expected_output=f"Weather data for all cities.", agent=weather_researcher,
)
safety_task = Task(
    description=f"Get safety for: {cities_str}.",
    expected_output=f"Safety data for all cities.", agent=safety_researcher,
)
report_task = Task(
    description=f"Create HierarchicalReport for {cities_str} using weather + safety research.",
    expected_output="A complete HierarchicalReport.",
    agent=report_writer, context=[weather_task, safety_task],
    output_pydantic=HierarchicalReport,
)

crew = Crew(
    agents=[weather_researcher, safety_researcher, report_writer],
    tasks=[weather_task, safety_task, report_task],
    process=Process.hierarchical,   # ← key: manager LLM orchestrates
    manager_llm=gemini,             # ← manager is a separate LLM call
    verbose=False,
)
print("Hierarchical crew ready — Process.hierarchical + manager_llm=gemini")

In [ ]:
result = crew.kickoff()
report: HierarchicalReport = result.pydantic

if report:
    for section in report.city_sections:
        print(f"### {section.city}")
        print(f"  Weather: {section.weather_summary}")
        print(f"  Safety:  {section.safety_summary}")
        print(f"  Rec:     {section.recommendation}\n")
    print(f"## Executive Summary\n{report.executive_summary}")
    print(f"\nTop City: {report.top_city}")
else:
    print(result.raw)

## Key Takeaways

| What You Saw | Why It Matters |
|---|---|
| `Process.hierarchical` | CrewAI injects a manager agent — you don't write manager logic |
| `manager_llm=gemini` | Manager is a separate LLM call that plans + delegates |
| Workers focus on their tools | Clean separation — each agent only knows its domain |

### CrewAI vs LangGraph for Hierarchical
| | CrewAI | LangGraph |
|---|---|---|
| Manager | Auto-injected by framework | Explicit supervisor node |
| Delegation | Manager LLM decides | Routing function decides |
| Visibility | Less transparent | Full state trace |
| Complexity | Simpler setup | More explicit control |

### Next: [ADK Hierarchical →](../ADK/hierarchical.ipynb)